# Predictive Modeling and Ablation Study

## Objective

This notebook evaluates the impact of contextual data on predictive maintenance performance.

Experiments:

1. Telemetry Only
2. Telemetry + Context

Performance will be compared using Macro F1 Score.

In [79]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier

from sklearn.metrics import f1_score, classification_report

In [80]:
df = pd.read_csv("../data/processed/context_fused_validated.csv")

print(df.shape)

df.head()

(9988, 52)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,Torque [Nm]_change,Tool wear [min]_change,ambient_humidity,energy_load_index,production_demand,days_since_maintenance,shift,torque_x_load,wear_x_demand,temp_x_humidity
0,13,M14872,M,298.6,309.1,1339,51.1,34,0,0,...,6.8,5.0,69.967142,57.439020,71.116047,28,0,2935.133910,2417.945598,21626.843447
1,14,M14873,M,298.6,309.2,1742,30.0,37,0,0,...,-21.1,3.0,63.617357,54.014745,52.663169,165,0,1620.442350,1948.537242,19670.486781
2,15,L47194,L,298.6,309.2,2035,19.6,40,0,0,...,-10.4,3.0,71.476885,84.159069,121.935583,64,1,1649.517759,4877.423327,22100.652960
3,16,L47195,L,298.6,309.2,1542,48.4,42,0,0,...,28.8,2.0,80.230299,40.470695,52.000516,55,0,1958.781651,2184.021691,24807.208316
4,17,M14876,M,298.6,309.2,1311,46.6,44,0,0,...,-1.8,2.0,62.658466,72.353832,147.340261,29,1,3371.688554,6482.971479,19373.997765


In [81]:
df.columns = [re.sub(r"[^A-Za-z0-9_]", "_", col) for col in df.columns]

df.columns.tolist()

['UDI',
 'Product_ID',
 'Type',
 'Air_temperature__K_',
 'Process_temperature__K_',
 'Rotational_speed__rpm_',
 'Torque__Nm_',
 'Tool_wear__min_',
 'Machine_failure',
 'TWF',
 'HDF',
 'PWF',
 'OSF',
 'RNF',
 'Air_temperature__K__roll_mean',
 'Process_temperature__K__roll_mean',
 'Rotational_speed__rpm__roll_mean',
 'Torque__Nm__roll_mean',
 'Tool_wear__min__roll_mean',
 'Air_temperature__K__roll_std',
 'Process_temperature__K__roll_std',
 'Rotational_speed__rpm__roll_std',
 'Torque__Nm__roll_std',
 'Tool_wear__min__roll_std',
 'Air_temperature__K__roll_var',
 'Process_temperature__K__roll_var',
 'Rotational_speed__rpm__roll_var',
 'Torque__Nm__roll_var',
 'Tool_wear__min__roll_var',
 'Air_temperature__K__lag1',
 'Process_temperature__K__lag1',
 'Rotational_speed__rpm__lag1',
 'Torque__Nm__lag1',
 'Tool_wear__min__lag1',
 'Air_temperature__K__lag2',
 'Process_temperature__K__lag2',
 'Rotational_speed__rpm__lag2',
 'Torque__Nm__lag2',
 'Tool_wear__min__lag2',
 'Air_temperature__K__change

In [82]:
df.to_csv("../data/processed/context_fused_model_ready.csv", index=False)

print("Model-ready dataset saved.")

Model-ready dataset saved.


In [83]:
print(df.columns.tolist())

['UDI', 'Product_ID', 'Type', 'Air_temperature__K_', 'Process_temperature__K_', 'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'Machine_failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Air_temperature__K__roll_mean', 'Process_temperature__K__roll_mean', 'Rotational_speed__rpm__roll_mean', 'Torque__Nm__roll_mean', 'Tool_wear__min__roll_mean', 'Air_temperature__K__roll_std', 'Process_temperature__K__roll_std', 'Rotational_speed__rpm__roll_std', 'Torque__Nm__roll_std', 'Tool_wear__min__roll_std', 'Air_temperature__K__roll_var', 'Process_temperature__K__roll_var', 'Rotational_speed__rpm__roll_var', 'Torque__Nm__roll_var', 'Tool_wear__min__roll_var', 'Air_temperature__K__lag1', 'Process_temperature__K__lag1', 'Rotational_speed__rpm__lag1', 'Torque__Nm__lag1', 'Tool_wear__min__lag1', 'Air_temperature__K__lag2', 'Process_temperature__K__lag2', 'Rotational_speed__rpm__lag2', 'Torque__Nm__lag2', 'Tool_wear__min__lag2', 'Air_temperature__K__change', 'Process_temperature__K__change', 'R

In [84]:
y = df["Machine_failure"]

print(y.value_counts())

Machine_failure
0    9649
1     339
Name: count, dtype: int64


In [85]:
telemetry_cols = [
    col
    for col in df.columns
    if col
    not in [
        "Machine_failure",
        "Product_ID",
        "Type",
        # leakage columns
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF",
        # context columns
        "ambient_humidity",
        "energy_load_index",
        "production_demand",
        "days_since_maintenance",
        "shift",
        "torque_x_load",
        "wear_x_demand",
        "temp_x_humidity",
    ]
]

X_telemetry = df[telemetry_cols]

In [86]:
X_context = df.drop(
    columns=[
        "Machine_failure",
        "Product_ID",
        "Type",
        # leakage columns
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF",
    ]
)

In [87]:
print("Telemetry:", X_telemetry.shape)

print("Context:", X_context.shape)

Telemetry: (9988, 36)
Context: (9988, 44)


In [88]:
print(X_telemetry.select_dtypes(include="object").columns)

print(X_context.select_dtypes(include="object").columns)

Index([], dtype='str')
Index([], dtype='str')


In [89]:
print("Telemetry Shape:", X_telemetry.shape)
print("Context Shape:", X_context.shape)

ablation_summary = pd.DataFrame(
    {
        "Dataset": ["Telemetry Only", "Telemetry + Context"],
        "Features": [X_telemetry.shape[1], X_context.shape[1]],
    }
)

ablation_summary

Telemetry Shape: (9988, 36)
Context Shape: (9988, 44)


,Dataset,Features
0,Telemetry Only,36
1,Telemetry + Context,44


In [90]:
X_train_t, X_test_t, y_train, y_test = train_test_split(
    X_telemetry, y, test_size=0.20, stratify=y, random_state=42
)

X_train_c, X_test_c, _, _ = train_test_split(
    X_context, y, test_size=0.20, stratify=y, random_state=42
)

In [91]:
print(y_train.value_counts())

print(y_test.value_counts())

Machine_failure
0    7719
1     271
Name: count, dtype: int64
Machine_failure
0    1930
1      68
Name: count, dtype: int64


In [92]:
telemetry_model = LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1)

telemetry_model.fit(X_train_t, y_train)

,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [93]:
telemetry_preds = telemetry_model.predict(X_test_t)

telemetry_f1 = f1_score(y_test, telemetry_preds, average="macro")

print("Telemetry Macro F1:", telemetry_f1)

Telemetry Macro F1: 0.8770542801729344


In [94]:
context_model = LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1)

context_model.fit(X_train_c, y_train)

,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [95]:
context_preds = context_model.predict(X_test_c)

context_f1 = f1_score(y_test, context_preds, average="macro")

print("Context Macro F1:", context_f1)

Context Macro F1: 0.8805815065470061


In [96]:
baseline_df = pd.DataFrame(
    {
        "Model": ["Telemetry Only", "Telemetry + Context"],
        "Macro_F1": [telemetry_f1, context_f1],
    }
)

baseline_df

,Model,Macro_F1
0,Telemetry Only,0.877054
1,Telemetry + Context,0.880582


In [97]:
baseline_df.to_csv("../reports/baseline_results.csv", index=False)

baseline_df

,Model,Macro_F1
0,Telemetry Only,0.877054
1,Telemetry + Context,0.880582


In [98]:
improvement = context_f1 - telemetry_f1

print(f"Context Improvement: {improvement:.4f}")

Context Improvement: 0.0035


### Baseline Ablation Results

Two baseline LightGBM models were evaluated.

| Model | Macro F1 |
|---------|---------:|
| Telemetry Only | 0.8771 |
| Telemetry + Context | 0.8806 |

The contextual feature set produced a modest improvement in predictive performance (+0.0035 Macro F1), indicating that external operational information contributes additional predictive signal beyond telemetry alone.

Further gains are expected through class imbalance handling, cross-validation, and model tuning.